## cui-tf-idf

In [1]:
import ast
import heapq
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# 1. Load existing CUI sets

human_df = pd.read_pickle("metadata/human_with_cuis.pkl")
mouse_df = pd.read_pickle("metadata/mouse_with_cuis.pkl")

# print(human_df.columns)
# print(mouse_df.columns)

# print(len(human_df), len(mouse_df))

Index(['gse_id', 'species', 'title', 'summary', 'text', 'cui_set'], dtype='str')
Index(['gse_id', 'species', 'title', 'summary', 'text', 'cui_set'], dtype='str')
3395 4066


In [3]:
# 2. Clean CUI set format

# def ensure_cui_set(x):
#     if isinstance(x, set):
#         return x
#     if isinstance(x, list):
#         return set(x)
#     if isinstance(x, str):
#         try:
#             return set(ast.literal_eval(x))
#         except Exception:
#             return set()
#     return set()

# human_df["cui_set"] = human_df["cui_set"].apply(ensure_cui_set)
# mouse_df["cui_set"] = mouse_df["cui_set"].apply(ensure_cui_set)

In [4]:
# 3. Convert CUI set to “CUI document”

human_df["cui_doc"] = human_df["cui_set"].apply(lambda s: " ".join(sorted(s)))
mouse_df["cui_doc"] = mouse_df["cui_set"].apply(lambda s: " ".join(sorted(s)))

In [ ]:
# 4. Fit TF-IDF on human + mouse together

all_cui_docs = pd.concat(
    [human_df["cui_doc"], mouse_df["cui_doc"]],
    ignore_index=True
)

vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    binary=True,
    use_idf=True,
    smooth_idf=True,
    norm="l2"
)

X_all = vectorizer.fit_transform(all_cui_docs)

n_human = len(human_df)

X_human = X_all[:n_human]
X_mouse = X_all[n_human:]

# print(X_human.shape)
# print(X_mouse.shape)

(3395, 33968)
(4066, 33968)


In [ ]:
# 5. Optional: inspect highest / lowest IDF CUIs

# idf = dict(zip(vectorizer.get_feature_names_out(), vectorizer.idf_))

# idf_df = pd.DataFrame({
#     "cui": list(idf.keys()),
#     "idf": list(idf.values())
# }).sort_values("idf", ascending=False)

# idf_df.head(20)

,cui,idf
0,C0000096,9.224432
17908,C1419998,9.224432
17905,C1419995,9.224432
17901,C1419991,9.224432
17898,C1419986,9.224432
17896,C1419983,9.224432
17892,C1419977,9.224432
17890,C1419945,9.224432
17888,C1419908,9.224432
17886,C1419894,9.224432


In [7]:
# 6. Helper: shared CUIs and shared IDF weight

def shared_cuis(a, b):
    return sorted(a & b)


def shared_idf_sum(cuis):
    return sum(idf.get(cui, 0) for cui in cuis)

In [8]:
# 7. Across species: human × mouse top pairs

output_path = Path("similarity-results/top_1000_human_mouse_cui_tfidf_pairs.csv")
top_n = 1000
chunk_size = 100

if output_path.exists():
    top_human_mouse_tfidf = pd.read_csv(output_path)
    print(f"Loaded existing file: {output_path}")

else:
    heap = []
    counter = 0

    for start in tqdm(range(0, X_human.shape[0], chunk_size), desc="human × mouse"):
        end = min(start + chunk_size, X_human.shape[0])

        sim_chunk = X_human[start:end] @ X_mouse.T
        sim_chunk = sim_chunk.toarray()

        flat = sim_chunk.ravel()

        candidate_n = min(top_n, flat.size)
        candidate_idx = np.argpartition(flat, -candidate_n)[-candidate_n:]

        for idx in candidate_idx:
            local_i, j = np.unravel_index(idx, sim_chunk.shape)
            i = start + local_i
            score = sim_chunk[local_i, j]

            if score <= 0:
                continue

            h_cuis = human_df.iloc[i]["cui_set"]
            m_cuis = mouse_df.iloc[j]["cui_set"]
            shared = shared_cuis(h_cuis, m_cuis)

            row = {
                "comparison": "human_mouse",
                "human_gse": human_df.iloc[i]["gse_id"],
                "mouse_gse": mouse_df.iloc[j]["gse_id"],
                "tfidf_cosine": score,
                "shared_cui_count": len(shared),
                "shared_idf_sum": shared_idf_sum(shared),
                "shared_cuis": shared,
                "human_title": human_df.iloc[i]["title"],
                "mouse_title": mouse_df.iloc[j]["title"],
            }

            item = (score, len(shared), shared_idf_sum(shared), counter, row)
            counter += 1

            if len(heap) < top_n:
                heapq.heappush(heap, item)
            else:
                heapq.heappushpop(heap, item)

    rows = [item[-1] for item in heap]

    top_human_mouse_tfidf = (
        pd.DataFrame(rows)
        .sort_values(
            ["tfidf_cosine", "shared_idf_sum", "shared_cui_count"],
            ascending=False
        )
        .reset_index(drop=True)
    )

    top_human_mouse_tfidf.to_csv(output_path, index=False)
    print(f"Saved new file: {output_path}")

top_human_mouse_tfidf.head(5)

Loaded existing file: similarity-results/top_1000_human_mouse_cui_tfidf_pairs.csv


,comparison,human_gse,mouse_gse,tfidf_cosine,shared_cui_count,shared_idf_sum,shared_cuis,human_title,mouse_title
0,human_mouse,GSE103658,GSE103725,1.0,101,496.886869,"['C0001792', 'C0002976', 'C0015609', 'C0017262...",Expression changes in Melanomas pre MAPKi trea...,Expression changes in Melanomas pre MAPKi trea...
1,human_mouse,GSE81148,GSE81149,1.0,93,466.537984,"['C0007593', 'C0007600', 'C0008546', 'C0008976...",RNASeq of MV4;11 cells transduced with scrambl...,RNASeq of MLL-AF9 cells transduced with scraml...
2,human_mouse,GSE105137,GSE105138,1.0,23,103.177077,"['C0001272', 'C0205216', 'C0334094', 'C0392756...",Gene Expression Profiling of melanoma cell lin...,Gene Expression Profiling of one melanoma cell...
3,human_mouse,GSE81328,GSE81329,1.0,21,115.663483,"['C0005961', 'C0005976', 'C0023467', 'C0026336...",MEIS2 is a novel oncogenic partner in AML1-ETO...,MEIS2 is a novel oncogenic partner in AML1-ETO...
4,human_mouse,GSE109702,GSE67934,1.0,2,16.839425,"['C0013862', 'C1704732']",Title not found on the page.,Title not found on the page.


In [9]:
# 8. Within human top pairs

output_path = Path("similarity-results/top_1000_within_human_cui_tfidf_pairs.csv")
top_n = 1000
chunk_size = 100

if output_path.exists():
    top_within_human_tfidf = pd.read_csv(output_path)
    print(f"Loaded existing file: {output_path}")

else:
    heap = []
    counter = 0
    n = X_human.shape[0]

    for start in tqdm(range(0, n, chunk_size), desc="within human"):
        end = min(start + chunk_size, n)

        sim_chunk = X_human[start:end] @ X_human.T
        sim_chunk = sim_chunk.toarray()

        for local_i in range(sim_chunk.shape[0]):
            i = start + local_i

            # avoid self-pair and duplicated reverse pair
            sim_chunk[local_i, :i + 1] = 0

        flat = sim_chunk.ravel()

        candidate_n = min(top_n, flat.size)
        candidate_idx = np.argpartition(flat, -candidate_n)[-candidate_n:]

        for idx in candidate_idx:
            local_i, j = np.unravel_index(idx, sim_chunk.shape)
            i = start + local_i
            score = sim_chunk[local_i, j]

            if score <= 0:
                continue

            cui_i = human_df.iloc[i]["cui_set"]
            cui_j = human_df.iloc[j]["cui_set"]
            shared = shared_cuis(cui_i, cui_j)

            row = {
                "comparison": "within_human",
                "gse_1": human_df.iloc[i]["gse_id"],
                "gse_2": human_df.iloc[j]["gse_id"],
                "tfidf_cosine": score,
                "shared_cui_count": len(shared),
                "shared_idf_sum": shared_idf_sum(shared),
                "shared_cuis": shared,
                "title_1": human_df.iloc[i]["title"],
                "title_2": human_df.iloc[j]["title"],
            }

            item = (score, len(shared), shared_idf_sum(shared), counter, row)
            counter += 1

            if len(heap) < top_n:
                heapq.heappush(heap, item)
            else:
                heapq.heappushpop(heap, item)

    rows = [item[-1] for item in heap]

    top_within_human_tfidf = (
        pd.DataFrame(rows)
        .sort_values(
            ["tfidf_cosine", "shared_idf_sum", "shared_cui_count"],
            ascending=False
        )
        .reset_index(drop=True)
    )

    top_within_human_tfidf.to_csv(output_path, index=False)
    print(f"Saved new file: {output_path}")

top_within_human_tfidf.head(5)

Loaded existing file: similarity-results/top_1000_within_human_cui_tfidf_pairs.csv


,comparison,gse_1,gse_2,tfidf_cosine,shared_cui_count,shared_idf_sum,shared_cuis,title_1,title_2
0,within_human,GSE94528,GSE94999,1.0,109,564.937214,"['C0001272', 'C0001688', 'C0004561', 'C0006826...","H3B-8800, a novel oral splicing modulator, ind...","H3B-8800, a novel oral splicing modulator, ind..."
1,within_human,GSE81074,GSE81080,1.0,108,547.524853,"['C0002793', 'C0002874', 'C0004561', 'C0005839...",Differentiation of human embryonic stem cells ...,Differentiation of human embryonic stem cells ...
2,within_human,GSE96562,GSE96563,1.0,149,773.952612,"['C0003320', 'C0003334', 'C0003341', 'C0004561...",Single cell RNA-seq reveals expansion of IGRP-...,Single cell RNA-seq reveals expansion of IGRP-...
3,within_human,GSE81497,GSE81498,1.0,133,657.067069,"['C0002684', 'C0003015', 'C0005495', 'C0006826...",Multiple mechanisms disrupt let-7 miRNA biogen...,Multiple mechanisms disrupt let-7 miRNA biogen...
4,within_human,GSE71456,GSE74953,1.0,116,608.618848,"['C0002793', 'C0007589', 'C0007600', 'C0007634...",Derivation and differentiation of haploid huma...,Derivation and differentiation of haploid huma...


In [10]:
# 9. Within mouse top pairs

output_path = Path("similarity-results/top_1000_within_mouse_cui_tfidf_pairs.csv")
top_n = 1000
chunk_size = 100

if output_path.exists():
    top_within_mouse_tfidf = pd.read_csv(output_path)
    print(f"Loaded existing file: {output_path}")

else:
    heap = []
    counter = 0
    n = X_mouse.shape[0]

    for start in tqdm(range(0, n, chunk_size), desc="within mouse"):
        end = min(start + chunk_size, n)

        sim_chunk = X_mouse[start:end] @ X_mouse.T
        sim_chunk = sim_chunk.toarray()

        for local_i in range(sim_chunk.shape[0]):
            i = start + local_i

            # avoid self-pair and duplicated reverse pair
            sim_chunk[local_i, :i + 1] = 0

        flat = sim_chunk.ravel()

        candidate_n = min(top_n, flat.size)
        candidate_idx = np.argpartition(flat, -candidate_n)[-candidate_n:]

        for idx in candidate_idx:
            local_i, j = np.unravel_index(idx, sim_chunk.shape)
            i = start + local_i
            score = sim_chunk[local_i, j]

            if score <= 0:
                continue

            cui_i = mouse_df.iloc[i]["cui_set"]
            cui_j = mouse_df.iloc[j]["cui_set"]
            shared = shared_cuis(cui_i, cui_j)

            row = {
                "comparison": "within_mouse",
                "gse_1": mouse_df.iloc[i]["gse_id"],
                "gse_2": mouse_df.iloc[j]["gse_id"],
                "tfidf_cosine": score,
                "shared_cui_count": len(shared),
                "shared_idf_sum": shared_idf_sum(shared),
                "shared_cuis": shared,
                "title_1": mouse_df.iloc[i]["title"],
                "title_2": mouse_df.iloc[j]["title"],
            }

            item = (score, len(shared), shared_idf_sum(shared), counter, row)
            counter += 1

            if len(heap) < top_n:
                heapq.heappush(heap, item)
            else:
                heapq.heappushpop(heap, item)

    rows = [item[-1] for item in heap]

    top_within_mouse_tfidf = (
        pd.DataFrame(rows)
        .sort_values(
            ["tfidf_cosine", "shared_idf_sum", "shared_cui_count"],
            ascending=False
        )
        .reset_index(drop=True)
    )

    top_within_mouse_tfidf.to_csv(output_path, index=False)
    print(f"Saved new file: {output_path}")

top_within_mouse_tfidf.head(5)

Loaded existing file: similarity-results/top_1000_within_mouse_cui_tfidf_pairs.csv


,comparison,gse_1,gse_2,tfidf_cosine,shared_cui_count,shared_idf_sum,shared_cuis,title_1,title_2
0,within_mouse,GSE101623,GSE101624,1.0,165,965.694778,"['C0001554', 'C0001563', 'C0005884', 'C0014457...",The neuropeptide Neuromedin U stimulates innat...,The neuropeptide Neuromedin U stimulates innat...
1,within_mouse,GSE76864,GSE76865,1.0,109,622.012226,"['C0001688', 'C0003241', 'C0003242', 'C0004561...",Independent roles of switching and hypermutati...,Independent roles of switching and hypermutati...
2,within_mouse,GSE73559,GSE73560,1.0,110,565.120041,"['C0004561', 'C0007634', 'C0011155', 'C0011377...",Gene expression analysis to identify Klf2 targ...,Gene expression analysis to identify Klf2 targ...
3,within_mouse,GSE77736,GSE77740,1.0,103,539.182528,"['C0001272', 'C0001779', 'C0001811', 'C0004561...",Single Novel single cell assay reveals progres...,Single Novel single cell assay reveals progres...
4,within_mouse,GSE103185,GSE104325,1.0,85,417.046933,"['C0001779', 'C0004909', 'C0006104', 'C0007613...",Early-life gene expression in neurons modulate...,Early-life gene expression in neurons modulate...


In [11]:
# 10. Combine all TF-IDF weighted results

human_mouse_clean = top_human_mouse_tfidf.rename(columns={
    "human_gse": "gse_1",
    "mouse_gse": "gse_2",
    "human_title": "title_1",
    "mouse_title": "title_2"
})

all_tfidf_pairs = pd.concat(
    [
        human_mouse_clean,
        top_within_human_tfidf,
        top_within_mouse_tfidf
    ],
    ignore_index=True
)

all_tfidf_pairs = all_tfidf_pairs.sort_values(
    ["tfidf_cosine", "shared_idf_sum", "shared_cui_count"],
    ascending=False
).reset_index(drop=True)

all_tfidf_pairs.to_csv("similarity-results/top_all_cui_tfidf_similarity_pairs.csv", index=False)

# all_tfidf_pairs.head(30)

In [12]:
# 11. Add metadata + remove exact matches + identify same publication/source

import ast
import re
import numpy as np
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 11. Add metadata, remove exact matches, and flag same source
# ------------------------------------------------------------

# input from previous step
pairs = all_tfidf_pairs.copy()

# If you already made this:
# all_tfidf_pairs_no_exact = ...
# then use:
# pairs = all_tfidf_pairs_no_exact.copy()


# ------------------------------------------------------------
# 11.1 Standardize pair columns
# ------------------------------------------------------------

# human_mouse rows should already have gse_1 / gse_2 after your combine step.
# gse_1 = human GSE for human_mouse
# gse_2 = mouse GSE for human_mouse

needed_cols = {"comparison", "gse_1", "gse_2", "tfidf_cosine", "shared_cuis"}
missing = needed_cols - set(pairs.columns)

if missing:
    raise ValueError(f"Missing columns in pairs dataframe: {missing}")

In [13]:
# ------------------------------------------------------------
# 11.2 Prepare metadata lookup from existing human_df / mouse_df
# ------------------------------------------------------------

def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"[^\w\s]", "", x)
    return x.strip()


def ensure_cui_set(x):
    if isinstance(x, set):
        return x
    if isinstance(x, list):
        return set(x)
    if isinstance(x, str):
        try:
            return set(ast.literal_eval(x))
        except Exception:
            return set()
    return set()


human_meta = human_df.copy()
mouse_meta = mouse_df.copy()

human_meta["cui_set"] = human_meta["cui_set"].apply(ensure_cui_set)
mouse_meta["cui_set"] = mouse_meta["cui_set"].apply(ensure_cui_set)

human_lookup = human_meta.set_index("gse_id").to_dict(orient="index")
mouse_lookup = mouse_meta.set_index("gse_id").to_dict(orient="index")

all_lookup = {**human_lookup, **mouse_lookup}


def get_meta_value(gse, field):
    record = all_lookup.get(gse, {})
    return record.get(field, np.nan)


# add original metadata
pairs["title_1_full"] = pairs["gse_1"].apply(lambda g: get_meta_value(g, "title"))
pairs["title_2_full"] = pairs["gse_2"].apply(lambda g: get_meta_value(g, "title"))

pairs["summary_1"] = pairs["gse_1"].apply(lambda g: get_meta_value(g, "summary"))
pairs["summary_2"] = pairs["gse_2"].apply(lambda g: get_meta_value(g, "summary"))

pairs["summary_1_norm"] = pairs["summary_1"].apply(normalize_text)
pairs["summary_2_norm"] = pairs["summary_2"].apply(normalize_text)

pairs["same_summary"] = pairs["summary_1_norm"] == pairs["summary_2_norm"]

In [14]:
# ------------------------------------------------------------
# 11.3 Identify exact same CUI-set pairs
# ------------------------------------------------------------

def get_cui_set_for_gse(gse):
    record = all_lookup.get(gse, {})
    return ensure_cui_set(record.get("cui_set", set()))


pairs["cui_set_1"] = pairs["gse_1"].apply(get_cui_set_for_gse)
pairs["cui_set_2"] = pairs["gse_2"].apply(get_cui_set_for_gse)

pairs["same_cui_set"] = pairs["cui_set_1"] == pairs["cui_set_2"]

print("Exact same CUI-set pairs:", pairs["same_cui_set"].sum())
print("Exact same summary pairs:", pairs["same_summary"].sum())

Exact same CUI-set pairs: 156
Exact same summary pairs: 452


In [15]:
# ------------------------------------------------------------
# 11.4 Load GEO source / PMID / subseries data
# ------------------------------------------------------------

# Expected files generated by find_pmid.py:
# metadata/gse_pmid_subseries_of_human.tsv
# metadata/gse_pmid_subseries_of_mouse.tsv

human_source_path = Path("metadata/gse_pmid_subseries_of_human.tsv")
mouse_source_path = Path("metadata/gse_pmid_subseries_of_mouse.tsv")

if human_source_path.exists() and mouse_source_path.exists():
    human_source = pd.read_csv(human_source_path, sep="\t")
    mouse_source = pd.read_csv(mouse_source_path, sep="\t")

    source_df = pd.concat([human_source, mouse_source], ignore_index=True)
    source_lookup = source_df.set_index("gse").to_dict(orient="index")

    print("Loaded source metadata.")
    print("Human source rows:", len(human_source))
    print("Mouse source rows:", len(mouse_source))

else:
    source_lookup = {}
    print("Source metadata files not found.")
    print("Expected:")
    print(human_source_path)
    print(mouse_source_path)
    print("Run the GEO source lookup script first, then rerun this cell.")


def source_value(gse, field):
    record = source_lookup.get(gse, {})
    value = record.get(field, np.nan)

    if isinstance(value, float) and pd.isna(value):
        return np.nan

    return value


def split_source_field(x):
    if pd.isna(x):
        return set()
    if isinstance(x, list):
        return set(map(str, x))
    return set(i.strip() for i in str(x).split(";") if i.strip())


def source_overlap(gse1, gse2, field):
    v1 = split_source_field(source_value(gse1, field))
    v2 = split_source_field(source_value(gse2, field))
    return sorted(v1 & v2)


# add source columns
for field in ["pmid", "subseries", "superseries", "affiliation", "BioProject", "SRA"]:
    pairs[f"{field}_1"] = pairs["gse_1"].apply(lambda g: source_value(g, field))
    pairs[f"{field}_2"] = pairs["gse_2"].apply(lambda g: source_value(g, field))
    pairs[f"same_{field}"] = pairs.apply(
        lambda r: len(source_overlap(r["gse_1"], r["gse_2"], field)) > 0,
        axis=1
    )
    pairs[f"shared_{field}"] = pairs.apply(
        lambda r: source_overlap(r["gse_1"], r["gse_2"], field),
        axis=1
    )


# ------------------------------------------------------------
# 11.5 Relationship label, matching the logic from find_pmid.py
# ------------------------------------------------------------

def source_relationship(row):
    same_pmid = row["same_pmid"]
    same_subseries = row["same_subseries"]
    same_sra = row["same_SRA"]
    same_bioproject = row["same_BioProject"]

    if same_pmid and same_subseries:
        return pd.Series({
            "same_source_prediction": True,
            "same_source_confidence": "Very High"
        })

    elif same_pmid or same_subseries:
        return pd.Series({
            "same_source_prediction": True,
            "same_source_confidence": "High"
        })

    elif same_sra or same_bioproject:
        return pd.Series({
            "same_source_prediction": True,
            "same_source_confidence": "Moderate"
        })

    else:
        return pd.Series({
            "same_source_prediction": False,
            "same_source_confidence": "Unknown"
        })


pairs[["same_source_prediction", "same_source_confidence"]] = pairs.apply(
    source_relationship,
    axis=1
)


print("Same-source prediction counts:")
display(pairs["same_source_confidence"].value_counts(dropna=False))

Loaded source metadata.
Human source rows: 3395
Mouse source rows: 4064
Same-source prediction counts:


same_source_confidence
Unknown      2237
Very High     578
High          185
Name: count, dtype: int64

In [16]:
# ------------------------------------------------------------
# 11.6 Remove exact same CUI-set and exact same summary pairs
# ------------------------------------------------------------

before = len(pairs)

excluded_exact = pairs[
    pairs["same_cui_set"] | pairs["same_summary"]
].copy()

filtered_pairs = pairs[
    ~(pairs["same_cui_set"] | pairs["same_summary"])
].copy().reset_index(drop=True)

after = len(filtered_pairs)

print(f"Pairs before filtering: {before}")
print(f"Excluded exact CUI-set or exact summary pairs: {before - after}")
print(f"Pairs after filtering: {after}")


# ------------------------------------------------------------
# 11.7 Save outputs
# ------------------------------------------------------------

pairs.to_csv(
    "similarity-results/top_all_cui_tfidf_similarity_pairs_with_metadata_source_flags.csv",
    index=False
)

excluded_exact.to_csv(
    "similarity-results/excluded_exact_cui_or_summary_pairs.csv",
    index=False
)

filtered_pairs.to_csv(
    "similarity-results/top_all_cui_tfidf_similarity_pairs_filtered_exact.csv",
    index=False
)

print("Saved:")
print("similarity-results/top_all_cui_tfidf_similarity_pairs_with_metadata_source_flags.csv")
print("similarity-results/excluded_exact_cui_or_summary_pairs.csv")
print("similarity-results/top_all_cui_tfidf_similarity_pairs_filtered_exact.csv")

Pairs before filtering: 3000
Excluded exact CUI-set or exact summary pairs: 470
Pairs after filtering: 2530
Saved:
similarity-results/top_all_cui_tfidf_similarity_pairs_with_metadata_source_flags.csv
similarity-results/excluded_exact_cui_or_summary_pairs.csv
similarity-results/top_all_cui_tfidf_similarity_pairs_filtered_exact.csv


In [17]:
# ------------------------------------------------------------
# 11.8 Quick view
# ------------------------------------------------------------

# filtered_pairs[[
#     "comparison",
#     "gse_1",
#     "gse_2",
#     "tfidf_cosine",
#     "shared_cui_count",
#     "shared_idf_sum",
#     "same_cui_set",
#     "same_summary",
#     "same_source_prediction",
#     "same_source_confidence",
#     "shared_pmid",
#     "shared_subseries",
#     "shared_BioProject",
#     "shared_SRA",
#     "title_1_full",
#     "title_2_full"
# ]].head(30)

human_mouse_pairs = filtered_pairs[filtered_pairs['comparison'] == 'human_mouse']

In [18]:
human_mouse_pairs[human_mouse_pairs['same_source_prediction'] == False]

,comparison,gse_1,gse_2,tfidf_cosine,shared_cui_count,shared_idf_sum,shared_cuis,title_1,title_2,title_1_full,...,BioProject_1,BioProject_2,same_BioProject,shared_BioProject,SRA_1,SRA_2,same_SRA,shared_SRA,same_source_prediction,same_source_confidence
75,human_mouse,GSE50582,GSE49844,0.865189,54,251.591338,"['C0002778', 'C0004793', 'C0006675', 'C0006754...",Transcriptomics analysis of gene expression in...,Transcriptomics analysis of gene expression in...,Transcriptomics analysis of gene expression in...,...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP029...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP028...,False,[],False,Unknown
141,human_mouse,GSE71095,GSE49844,0.784066,54,251.591338,"['C0002778', 'C0004793', 'C0006675', 'C0006754...",Transcriptomics analysis of gene expression in...,Transcriptomics analysis of gene expression in...,Transcriptomics analysis of gene expression in...,...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP061...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP028...,False,[],False,Unknown
162,human_mouse,GSE80237,GSE72941,0.754862,17,59.752372,"['C0002778', 'C0017260', 'C0017337', 'C0040649...",RNA-seq analysis reveals profound changes in t...,RNA-seq analysis reveals profound changes in t...,RNA-seq analysis reveals profound changes in t...,...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP073...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP063...,False,[],False,Unknown
183,human_mouse,GSE81478,GSE75714,0.724375,17,69.352139,"['C0023810', 'C0040649', 'C0175697', 'C0871261...",Transcriptome sequencing wide functional analy...,RNA Sequencing Facilitates Quantitative Analys...,Transcriptome sequencing wide functional analy...,...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP075...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP067...,False,[],False,Unknown
192,human_mouse,GSE50582,GSE53249,0.720648,47,208.999972,"['C0002778', 'C0004793', 'C0006675', 'C0006754...",Transcriptomics analysis of gene expression in...,Transcriptomics analysis of gene expression in...,Transcriptomics analysis of gene expression in...,...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP029...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP033...,False,[],False,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2525,human_mouse,GSE99180,GSE89198,0.306276,9,47.223386,"['C0007634', 'C0012582', 'C1136359', 'C1419059...","CD133hi, Notchhi, DP (double positive) and DN ...",BCL11B AND COMBINATORIAL RESOLUTION OF CELL FA...,"CD133hi, Notchhi, DP (double positive) and DN ...",...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP107...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP092...,False,[],False,Unknown
2526,human_mouse,GSE108003,GSE87870,0.306180,5,26.574831,"['C0023467', 'C0215508', 'C1335654', 'C1435548...",A machine learning approach to integrate big d...,Next gen RNA sequencing of mouse acute myeloid...,A machine learning approach to integrate big d...,...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP126...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP091...,False,[],False,Unknown
2527,human_mouse,GSE62526,GSE52868,0.306131,6,18.015662,"['C0752248', 'C1327760', 'C1449575', 'C1513400...",Gene expression profiling of melanoma cell lin...,Expression profiling of mouse bone ma

In [19]:
filtered_pairs[filtered_pairs['same_source_prediction'] == False]

,comparison,gse_1,gse_2,tfidf_cosine,shared_cui_count,shared_idf_sum,shared_cuis,title_1,title_2,title_1_full,...,BioProject_1,BioProject_2,same_BioProject,shared_BioProject,SRA_1,SRA_2,same_SRA,shared_SRA,same_source_prediction,same_source_confidence
4,within_mouse,GSE61031,GSE61033,0.995175,101,497.327250,"['C0001688', 'C0002684', 'C0011435', 'C0012634...",Dicer WT/KO MSC RNA-Seq [polyA RNA],Dicer WT/KO MSC RNA-Seq [total RNA],Dicer WT/KO MSC RNA-Seq [polyA RNA],...,NaN,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP045...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP046...,False,[],False,Unknown
10,within_mouse,GSE76458,GSE78151,0.985447,123,663.078958,"['C0000934', 'C0001779', 'C0004083', 'C0004461...",The transcriptome of central nervous system my...,The transcriptome of central nervous system my...,The transcriptome of central nervous system my...,...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP067...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP070...,False,[],False,Unknown
12,within_mouse,GSE80166,GSE81095,0.980661,71,330.632338,"['C0004561', 'C0007589', 'C0007600', 'C0007634...",RNA Sequencing Data in differentiating mouse e...,RNA Sequencing Data in differentiating mouse e...,RNA Sequencing Data in differentiating mouse e...,...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP073...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP074...,False,[],False,Unknown
13,within_mouse,GSE45119,GSE52856,0.979568,85,378.398766,"['C0002778', 'C0004561', 'C0007634', 'C0008300...",Gene Expression and Exon Splicing Change Analy...,Gene Expression and Exon Splicing Change Analy...,Gene Expression and Exon Splicing Change Analy...,...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,NaN,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP019...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP033...,False,[],False,Unknown
14,within_mouse,GSE75431,GSE93179,0.977374,50,254.801835,"['C0004112', 'C0004561', 'C0006104', 'C0007634...",Sorted cells_PS2APP brains_7/13mo,Tau-P301L sorted cell types,Sorted cells_PS2APP brains_7/13mo,...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP066...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP096...,False,[],False,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2525,human_mouse,GSE99180,GSE89198,0.306276,9,47.223386,"['C0007634', 'C0012582', 'C1136359', 'C1419059...","CD133hi, Notchhi, DP (double positive) and DN ...",BCL11B AND COMBINATORIAL RESOLUTION OF CELL FA...,"CD133hi, Notchhi, DP (double positive) and DN ...",...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP107...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP092...,False,[],False,Unknown
2526,human_mouse,GSE108003,GSE87870,0.306180,5,26.574831,"['C0023467', 'C0215508', 'C1335654', 'C1435548...",A machine learning approach to integrate big d...,Next gen RNA sequencing of mouse acute myeloid...,A machine learning approach to integrate big d...,...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https://www.ncbi.nlm.nih.gov/sra?term=SRP126...,['https://www.ncbi.nlm.nih.gov/sra?term=SRP091...,False,[],False,Unknown
2527,human_mouse,GSE62526,GSE52868,0.306131,6,18.015662,"['C0752248', 'C1327760', 'C1449575', 'C1513400...",Gene expression profiling of melanoma cell lin...,Expression profiling of mouse bone marrow pre-...,Gene expression profiling of melanoma cell lin...,...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,['https://www.ncbi.nlm.nih.gov/bioproject/PRJN...,False,[],['https:

## raw text similarity

In [20]:
# ------------------------------------------------------------
# 12. Text similarity vs CUI-TF-IDF similarity
# ------------------------------------------------------------

import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
# load
pairs = filtered_pairs.copy()

pairs = pairs.rename(columns={
    "tfidf_cosine": "cui_tfidf_cosine"
})

# normalize
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"\s+", " ", x)
    return x.strip()


pairs["title_1_norm"] = pairs["title_1_full"].apply(normalize_text)
pairs["title_2_norm"] = pairs["title_2_full"].apply(normalize_text)

pairs["summary_1_norm"] = pairs["summary_1"].apply(normalize_text)
pairs["summary_2_norm"] = pairs["summary_2"].apply(normalize_text)

pairs["text_1_norm"] = (
    pairs["title_1_norm"] + " " + pairs["summary_1_norm"]
).str.strip()

pairs["text_2_norm"] = (
    pairs["title_2_norm"] + " " + pairs["summary_2_norm"]
).str.strip()

In [22]:
# levenstein similarity (edit-distance)
from rapidfuzz.distance import DamerauLevenshtein


def damerau_similarity(a, b):
    a = normalize_text(a)
    b = normalize_text(b)

    max_len = max(len(a), len(b))

    if max_len == 0:
        return 0

    dist = DamerauLevenshtein.distance(a, b)
    return 1 - dist / max_len

pairs["title_damerau_sim"] = pairs.apply(
    lambda r: damerau_similarity(r["title_1_norm"], r["title_2_norm"]),
    axis=1
)

pairs["summary_damerau_sim"] = pairs.apply(
    lambda r: damerau_similarity(r["summary_1_norm"], r["summary_2_norm"]),
    axis=1
)

pairs["text_damerau_sim"] = pairs.apply(
    lambda r: damerau_similarity(r["text_1_norm"], r["text_2_norm"]),
    axis=1
)

In [23]:
# text tf-idf semantic similarity
def pairwise_text_tfidf_similarity(texts_1, texts_2):
    all_texts = pd.concat(
        [texts_1, texts_2],
        ignore_index=True
    )

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        min_df=1,
        ngram_range=(1, 2)
    )

    X = vectorizer.fit_transform(all_texts)

    n = len(texts_1)

    X1 = X[:n]
    X2 = X[n:]

    sims = cosine_similarity(X1, X2).diagonal()

    return sims

pairs["title_text_tfidf_cosine"] = pairwise_text_tfidf_similarity(
    pairs["title_1_norm"],
    pairs["title_2_norm"]
)

pairs["summary_text_tfidf_cosine"] = pairwise_text_tfidf_similarity(
    pairs["summary_1_norm"],
    pairs["summary_2_norm"]
)

pairs["text_tfidf_cosine"] = pairwise_text_tfidf_similarity(
    pairs["text_1_norm"],
    pairs["text_2_norm"]
)

In [24]:
# flag near-iidentical pairs
pairs["near_same_title"] = pairs["title_damerau_sim"] >= 0.90
pairs["near_same_summary"] = pairs["summary_damerau_sim"] >= 0.95
pairs["near_same_text"] = pairs["text_damerau_sim"] >= 0.95

# find disgareement btw approaches
cui_high_cutoff = pairs["cui_tfidf_cosine"].quantile(0.75)
text_high_cutoff = pairs["text_tfidf_cosine"].quantile(0.75)
edit_high_cutoff = pairs["text_damerau_sim"].quantile(0.75)

pairs["cui_high"] = pairs["cui_tfidf_cosine"] >= cui_high_cutoff
pairs["text_tfidf_high"] = pairs["text_tfidf_cosine"] >= text_high_cutoff
pairs["edit_high"] = pairs["text_damerau_sim"] >= edit_high_cutoff

def classify_pair(row):
    if row["cui_high"] and row["edit_high"]:
        return "high_cui_high_edit_near_duplicate"

    if row["cui_high"] and (not row["edit_high"]) and row["text_tfidf_high"]:
        return "high_cui_low_edit_high_text_semantic"

    if row["cui_high"] and (not row["edit_high"]) and (not row["text_tfidf_high"]):
        return "high_cui_low_text_potential_cui_specific"

    if (not row["cui_high"]) and row["text_tfidf_high"]:
        return "low_cui_high_text"

    return "other"


pairs["similarity_pattern"] = pairs.apply(classify_pair, axis=1)

pairs["similarity_pattern"].value_counts()

similarity_pattern
other                                       1695
high_cui_high_edit_near_duplicate            432
low_cui_high_text                            202
high_cui_low_text_potential_cui_specific     169
high_cui_low_edit_high_text_semantic          32
Name: count, dtype: int64

In [29]:
# score distribution
score_summary = pairs[[
    "cui_tfidf_cosine",
    "title_damerau_sim",
    "summary_damerau_sim",
    "text_damerau_sim",
    "title_text_tfidf_cosine",
    "summary_text_tfidf_cosine",
    "text_tfidf_cosine"
]].describe()

score_summary

,cui_tfidf_cosine,title_damerau_sim,summary_damerau_sim,text_damerau_sim,title_text_tfidf_cosine,summary_text_tfidf_cosine,text_tfidf_cosine
count,2530.000000,2530.000000,2530.000000,2530.000000,2530.000000,2530.000000,2530.000000
mean,0.437250,0.347982,0.343747,0.354715,0.150495,0.183684,0.193274
std,0.146316,0.204578,0.207199,0.186965,0.218565,0.241136,0.222172
min,0.306077,0.038835,0.033967,0.082949,0.000000,0.000000,0.002192
25%,0.341341,0.217391,0.223847,0.245685,0.007218,0.026468,0.040028
50%,0.384013,0.265976,0.264608,0.282353,0.055549,0.074556,0.097226
75%,0.464376,0.400000,0.377069,0.392335,0.197421,0.235778,0.263553
max,0.999112,1.000000,0.999186,0.999164,1.000000,1.000000,0.998622


In [28]:
# corr btw approaches
score_corr = pairs[[
    "cui_tfidf_cosine",
    "title_damerau_sim",
    "summary_damerau_sim",
    "text_damerau_sim",
    "title_text_tfidf_cosine",
    "summary_text_tfidf_cosine",
    "text_tfidf_cosine"
]].corr()

score_corr

,cui_tfidf_cosine,title_damerau_sim,summary_damerau_sim,text_damerau_sim,title_text_tfidf_cosine,summary_text_tfidf_cosine,text_tfidf_cosine
cui_tfidf_cosine,1.000000,0.595830,0.764184,0.796292,0.635234,0.798547,0.817085
title_damerau_sim,0.595830,1.000000,0.564015,0.699008,0.884572,0.584787,0.674329
summary_damerau_sim,0.764184,0.564015,1.000000,0.961619,0.534944,0.897539,0.836068
text_damerau_sim,0.796292,0.699008,0.961619,1.000000,0.653793,0.889444,0.873400
title_text_tfidf_cosine,0.635234,0.884572,0.534944,0.653793,1.000000,0.611026,0.747326
summary_text_tfidf_cosine,0.798547,0.584787,0.897539,0.889444,0.611026,1.000000,0.961167
text_tfidf_cosine,0.817085,0.674329,0.836068,0.873400,0.747326,0.961167,1.000000


In [31]:
# cases
# CUI high, text not near-identical
interesting_cui_pairs = (
    pairs[
        (pairs["cui_high"]) &
        (~pairs["near_same_summary"]) &
        (~pairs["near_same_text"])
    ]
    .sort_values("cui_tfidf_cosine", ascending=False)
)

interesting_cui_pairs[[
    "comparison",
    "gse_1",
    "gse_2",
    "cui_tfidf_cosine",
    "text_tfidf_cosine",
    "title_damerau_sim",
    "summary_damerau_sim",
    "same_source_prediction",
    "same_source_confidence",
    "title_1_full",
    "title_2_full"
]].head(5)

,comparison,gse_1,gse_2,cui_tfidf_cosine,text_tfidf_cosine,title_damerau_sim,summary_damerau_sim,same_source_prediction,same_source_confidence,title_1_full,title_2_full
8,within_human,GSE86977,GSE86985,0.986182,0.987146,0.923729,0.333190,True,High,REGION-SPECIFIC NEURAL STEM CELL LINEAGES REVE...,REGION-SPECIFIC NEURAL STEM CELL LINEAGES REVE...
12,within_mouse,GSE80166,GSE81095,0.980661,0.927718,0.656566,0.940048,False,Unknown,RNA Sequencing Data in differentiating mouse e...,RNA Sequencing Data in differentiating mouse e...
20,within_mouse,GSE95434,GSE95445,0.965987,0.744903,0.845455,0.873294,True,Very High,Single cell RNA-seq of 346 epithelial cells fr...,Single cell RNA-seq of 444 epithelial cells fr...
24,within_human,GSE100183,GSE69876,0.953409,0.867532,0.828571,0.852941,True,Very High,MYCL and EP400 are required for Max and MCPyV ...,EP400 is required for Max and MCPyV mediated g...
26,human_mouse,GSE37061,GSE37018,0.946151,0.854931,0.878788,0.930131,True,High,RNA-sequencing analysis of NB4 cells overexpre...,RNA-sequencing analysis of 32Dclone3 cells ove...


In [ ]:
# Text high but CUI relatively low
text_high_cui_lower = (
    pairs[
        (~pairs["cui_high"]) &
        (pairs["text_tfidf_high"])
    ]
    .sort_values("text_tfidf_cosine", ascending=False)
)

text_high_cui_lower[[
    "comparison",
    "gse_1",
    "gse_2",
    "cui_tfidf_cosine",
    "text_tfidf_cosine",
    "title_damerau_sim",
    "summary_damerau_sim",
    "title_1_full",
    "title_2_full"
]].head(5)

In [27]:
# save
output_path = Path("similarity-results/top_all_cui_tfidf_pairs_with_text_similarity.csv")

pairs.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

Saved: similarity-results/top_all_cui_tfidf_pairs_with_text_similarity.csv
